# 🎓 Student Wellbeing RAFT-RAG — Final v2 Cloud Demo

This notebook loads the trained v2 LoRA adapter from Google Drive and runs the complete local-first pipeline in Colab:

Student question → safety check → BGE-small → FAISS → 0.55 confidence gate → Qwen2.5-1.5B + v2 LoRA → concise grounded answer.


## 1. Mount Google Drive and verify the trained adapter


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

ADAPTER_PATH = Path("/content/drive/MyDrive/psychology-qwen-qlora-v2")
ADAPTER_FILE = ADAPTER_PATH / "adapter_model.safetensors"

print("Adapter folder exists:", ADAPTER_PATH.exists())
print("Adapter weights exist:", ADAPTER_FILE.exists())


## 2. Clone the latest repository


In [ ]:
%cd /content
!rm -rf psychology-raft-rag-chatbot
!git clone https://github.com/karthikeyan-crypto/psychology-raft-rag-chatbot.git
%cd /content/psychology-raft-rag-chatbot

!git status --short


## 3. Install inference dependencies


In [ ]:
!pip -q install -U "transformers>=5.0" "peft>=0.21" "bitsandbytes>=0.50" "accelerate>=1.0" "sentence-transformers>=3.1" "faiss-cpu>=1.8"


## 4. GPU check


In [ ]:
import torch
import transformers, peft, bitsandbytes as bnb

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("bitsandbytes:", bnb.__version__)


## 5. Build the local FAISS index


In [ ]:
!python inference/rag_local.py --source-dir data/sources --index-dir data/vector_index


In [ ]:
from inference.rag_local import LocalRetriever, DEFAULT_MIN_RETRIEVAL_SCORE

retriever = LocalRetriever("data/vector_index")

print("Minimum retrieval score:", DEFAULT_MIN_RETRIEVAL_SCORE)


## 6. Load clean Qwen base model


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

base_model.config.use_cache = False

print("Clean Qwen base model loaded.")
print("GPU:", torch.cuda.get_device_name(0))


## 7. Load the v2 LoRA adapter


In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_PATH),
    is_trainable=False,
)

model.eval()

print("v2 LoRA adapter loaded successfully.")
print("Active adapters:", model.active_adapters)


## 8. Safety + grounded generation


In [ ]:
MIN_RETRIEVAL_SCORE = 0.55
TOP_K = 3

SYSTEM_PROMPT = """You are a student wellbeing psychoeducation assistant.

STRICT GROUNDING RULES:
1. Use ONLY the retrieved knowledge as factual authority.
2. Every factual claim must be supported by the retrieved knowledge.
3. Do not add facts from pretrained knowledge.
4. Do not invent numbers, durations, examples, causes, mechanisms, or recommendations.
5. Do not diagnose mental-health conditions.
6. Do not claim to be a psychologist, therapist, psychiatrist, or doctor.
7. Do not recommend medication or personalised treatment.
8. If the retrieved knowledge is insufficient, say:
   "I don't have enough information in the current knowledge base to answer that reliably."

STYLE:
- Be concise and student-friendly.
- Give important points first.
- Prefer 4-7 short bullet points when several supported actions are relevant.
- Avoid repetition.
- Usually stay under 120 words.
"""

HIGH_RISK_TERMS = (
    "suicide",
    "kill myself",
    "end my life",
    "want to die",
    "self harm",
    "self-harm",
)

def safety_response(question: str):
    q = question.lower()
    if any(term in q for term in HIGH_RISK_TERMS):
        return (
            "I'm sorry you're dealing with something this serious. "
            "I can't provide emergency or clinical care. If you may act "
            "on thoughts of suicide or self-harm, contact local emergency "
            "services, go to the nearest emergency department, or stay "
            "with a trusted person who can help you get immediate human "
            "support. In India, Tele-MANAS is available at 14416 or "
            "1800-89-14416."
        )
    return None

def generate_answer(question: str):
    emergency = safety_response(question)
    if emergency:
        return emergency, []

    matches = retriever.search(question, top_k=TOP_K)

    strong_matches = [
        item for item in matches
        if float(item["score"]) >= MIN_RETRIEVAL_SCORE
    ]

    if not strong_matches:
        return (
            "I don't have enough information in the current knowledge "
            "base to answer that reliably.",
            matches,
        )

    context = "\n\n".join(
        f'<SOURCE name="{item["source"]}">\n{item["text"]}\n</SOURCE>'
        for item in strong_matches
    )

    user_prompt = f"""Retrieved knowledge:

{context}

Student question:
{question}

Answer ONLY from the retrieved knowledge.
Do not add outside facts.
"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=False,
    )

    input_device = next(model.parameters()).device
    model_inputs = {
        key: value.to(input_device)
        for key, value in model_inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            input_ids=model_inputs["input_ids"],
            attention_mask=model_inputs["attention_mask"],
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[
        0,
        model_inputs["input_ids"].shape[-1]:
    ]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return answer, strong_matches


## 9. Final validation


In [ ]:
tests = [
    "How can a student manage exam stress?",
    "What are some ways to improve sleep during stressful periods?",
    "What is the current stock price of NVIDIA?",
    "I want to kill myself.",
]

for question in tests:
    print("\n" + "=" * 80)
    print("QUESTION:", question)

    answer, matches = generate_answer(question)

    print("\nRETRIEVED SOURCES:")
    if matches:
        for item in matches:
            print(f"- {item['source']} | score={item['score']:.4f}")
    else:
        print("No sufficiently relevant sources.")

    print("\nANSWER:")
    print(answer)


## 10. Interactive judge demo


In [ ]:
print("Student Wellbeing Chatbot")
print("Type 'exit' to stop.")

while True:
    question = input("\nStudent: ").strip()

    if question.lower() in {"exit", "quit"}:
        break

    if not question:
        continue

    answer, matches = generate_answer(question)

    print("\nAssistant:", answer)

    if matches:
        print("\nSources used:")
        for item in matches:
            print(f"- {item['source']} | score={item['score']:.3f}")
    else:
        print("\nSources used: none above the confidence threshold.")
